# Los cinco niveles de agencia, explicados

**Sesión 1 · Agentes y arquitecturas multiagénticas**
Programa de formación en IA · Qypher para Grupo Bios

---

Este notebook existe para **leerse y correrse mientras se explica**. Todo el
código está a la vista: no importa nada del repositorio, no hay clases base ni
contratos de eventos, y las llamadas al modelo son síncronas. Cada nivel es una
función de veinte líneas que cabe en una pantalla.

| Nivel | Qué agrega respecto al anterior | Líneas |
|---|---|---|
| **N1** Procesador simple ☆☆☆ | nada: una llamada y ya | 4 |
| **N2** Enrutador ★☆☆ | el modelo **decide** algo | 6 |
| **N3** Llamador de herramientas ★★☆ | el modelo puede **pedir** que ejecutes una función | 18 |
| **N4** Agente multipasos ★★★ | **un `while`** alrededor de N3 | 20 |
| **N5** Sistema multiagente ★★★★ | dos N4 **expuestos como herramientas** de un tercero | 22 |

Fíjate en la última columna. La distancia entre N3 y N4 son **tres líneas**: un
`while`, un `break` y una indentación. Casi todo lo que se llama "framework de
agentes" es eso, más manejo de errores.

> ### ⚠ Datos sintéticos
>
> La "base de datos" de este notebook es un diccionario de Python inventado, con
> tres plantas y cuatro registros. Los nombres de municipio son reales; todo lo
> demás es ficticio y no representa la operación de Grupo Bios. Ningún dato real
> de la compañía se procesa en esta sesión.

### Los otros dos artefactos

| Artefacto | Para qué |
|---|---|
| **Este notebook** | Leer el código y entender cada nivel |
| **`2-los-cinco-niveles-taller.ipynb`** | Construirlos tú, con `# TODO` que completas |
| **El tablero** (`localhost:8000`) | Ver el flujo dibujado y comparar costos |

---

## 0 · Preparación

Dos celdas. La primera es todo lo que necesitas del mundo exterior.

In [ ]:
import json
import os

from openai import OpenAI

MODELO = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
cliente = OpenAI()  # toma la key de la variable de entorno OPENAI_API_KEY

# Cada llamada al modelo queda registrada acá. Al final del notebook la usamos
# para comparar cuánto cuesta cada nivel — que es la conclusión de la sesión.
REGISTRO = []


def preguntar(mensajes, tools=None, **extra):
    """Una llamada al modelo. TODO el notebook pasa por esta función.

    Es la única pieza de infraestructura que hay, y son seis líneas: se manda una
    lista de mensajes, opcionalmente una lista de herramientas declaradas, y
    vuelve la respuesta cruda del proveedor.
    """
    respuesta = cliente.chat.completions.create(
        model=MODELO,
        messages=mensajes,
        temperature=0,
        **({"tools": tools} if tools else {}),
        **extra,
    )
    REGISTRO.append(
        {
            "nivel": NIVEL_ACTUAL,
            "entrada": respuesta.usage.prompt_tokens,
            "salida": respuesta.usage.completion_tokens,
        }
    )
    return respuesta


NIVEL_ACTUAL = "—"  # solo para etiquetar el registro


def paso(texto, sangria=0):
    """Imprime lo que está ocurriendo. Sin esto, un agente es una caja negra."""
    print("   " * sangria + texto)


print(f"modelo: {MODELO}")
print(f"key   : {'presente' if os.getenv('OPENAI_API_KEY') else 'AUSENTE'}")


: 

### La "base de datos"

Quince líneas. A propósito: cuando la fuente de datos es un diccionario que cabe
en pantalla, **nadie puede confundir el mérito del agente con el mérito del SQL.**
Todo lo que veas funcionar sale de acá.

In [ ]:
INVENTARIO = {
    "Itagüí":  {"maíz": 320,  "soya": 180},   # ← 320 t: acuérdate de este número
    "Buga":    {"maíz": 1450, "soya": 900},
    "Palmira": {"maíz": 260,  "soya": 120},
}

# Toneladas de maíz que se necesitan esta semana en cada planta.
DEMANDA_SEMANAL = {"Itagüí": 1650, "Buga": 1200, "Palmira": 300}

EQUIPOS = {
    "molino-1":       {"planta": "Itagüí", "fallas_60d": 3, "vibracion": "creciente"},
    "peletizadora-1": {"planta": "Itagüí", "fallas_60d": 0, "vibracion": "estable"},
}

PEDIDOS = {
    "PD-871": {"planta": "Itagüí", "estado": "en muelle", "turno": 6, "pasos_restantes": 3},
}

PREGUNTA = "¿Cuánto maíz le queda a la planta de Itagüí y me alcanza para esta semana?"

print(f"Itagüí tiene {INVENTARIO['Itagüí']['maíz']} t de maíz")
print(f"y necesita {DEMANDA_SEMANAL['Itagüí']} t esta semana.")
print(f"→ La respuesta correcta es: NO alcanza, faltan "
      f"{DEMANDA_SEMANAL['Itagüí'] - INVENTARIO['Itagüí']['maíz']} t.")

---

## 1 · N1 · Procesador simple ☆☆☆

```python
process_llm_output(llm_response)
```

Una llamada al modelo. Sin herramientas, sin decisiones, sin acceso a nada.

**Lee la función antes de correrla.** Cuatro líneas: arma dos mensajes, llama, y
devuelve el texto. No hay ninguna vía por la que el modelo pueda enterarse de lo
que dice `INVENTARIO`.

In [ ]:
NIVEL_ACTUAL = "N1"


def n1(pregunta):
    respuesta = preguntar([
        {"role": "system", "content": "Eres un asistente de operaciones de una planta de alimentos."},
        {"role": "user", "content": pregunta},
    ])
    return respuesta.choices[0].message.content


print("EL MODELO DICE:")
print(" ", n1(PREGUNTA))
print()
print("LA REALIDAD:")
print(f"  {INVENTARIO['Itagüí']['maíz']} t de maíz · necesita {DEMANDA_SEMANAL['Itagüí']} t")

### 👀 Qué acaba de pasar

Una de dos, y las dos enseñan lo mismo:

- **Si dio una cifra**, no salió de ninguna parte. No es que se haya equivocado al
  consultar: es que *no consultó*. No tiene con qué.
- **Si se negó** («no tengo acceso a esos datos»), se portó bien — y sigue siendo
  inútil para la operación. Prueba a insistirle con *«dame un valor típico para una
  planta de este tamaño»* y mira qué pasa.

### 🔬 Tu turno

Cambia el `system` para prohibirle inventar: *«Si no tienes el dato, dilo y no
estimes.»* Vuelve a correr. ¿Cambia algo? El problema de N1 **no es el prompt**.

---

## 2 · N2 · Enrutador ★☆☆

```python
if llm_decision(): path_a() else: path_b()
```

Acá aparece lo que abre la puerta a los agentes: **el LLM deja de generar texto y
empieza a decidir.** Clasifica la pregunta en un dominio y ahí se detiene — no
consulta nada, no responde nada.

Dos líneas más que N1: una lista de opciones y una validación de la respuesta.

In [ ]:
NIVEL_ACTUAL = "N2"

DOMINIOS = ["mantenimiento", "compras", "logistica", "demanda"]


def n2(pregunta):
    respuesta = preguntar([
        {"role": "system", "content":
            f"Clasifica la pregunta del usuario en UNO de estos dominios: "
            f"{', '.join(DOMINIOS)}. Responde únicamente con la palabra, sin explicar."},
        {"role": "user", "content": pregunta},
    ])
    eleccion = respuesta.choices[0].message.content.strip().lower().rstrip(".")
    # Validar SIEMPRE: el modelo devuelve texto libre y puede contestar cualquier
    # cosa. En el taller esto se hace con salida estructurada, que es lo robusto.
    return eleccion if eleccion in DOMINIOS else f"(no reconocido: {eleccion!r})"


for p in [PREGUNTA,
          "El molino de Itagüí está vibrando raro",
          "¿Dónde está el pedido PD-871?"]:
    paso(f"{n2(p):<16} ← {p}")

### 👀 Qué acaba de pasar

Clasificó bien… y no sirvió de nada. **La pregunta que sigue es la puerta de
entrada a N3:** ¿y ahora quién ejecuta?

### 🔬 Tu turno

Prueba con *«El pedido de la avícola no llegó y creo que el molino está parado»*.
Toca dos dominios, y el router **tiene que elegir uno**. Con eso pierde la mitad
del problema — recuerda este momento cuando lleguemos a N5.

---

## 3 · N3 · Llamador de herramientas ★★☆

```python
run_function(llm_chosen_tool, llm_chosen_args)
```

**El nivel más importante de la clase.** Y la idea que hay que entender es una:

> El modelo **no ejecuta nada**. Solo produce un JSON que dice qué función
> ejecutarías y con qué argumentos. Ejecutarla es tu trabajo.

Necesitamos tres cosas: las funciones de Python, su descripción para el modelo, y
el bucle que une las dos.

In [ ]:
# ── 1 · Las funciones. Python normal, nada especial. ──────────────────────────

def consultar_inventario(planta, material):
    cantidad = INVENTARIO.get(planta, {}).get(material)
    if cantidad is None:
        return {"error": f"No tengo inventario de {material} en {planta}."}
    return {"planta": planta, "material": material, "toneladas": cantidad}


def consultar_demanda(planta):
    if planta not in DEMANDA_SEMANAL:
        return {"error": f"No tengo demanda registrada para {planta}."}
    return {"planta": planta, "toneladas_necesarias": DEMANDA_SEMANAL[planta]}


FUNCIONES = {
    "consultar_inventario": consultar_inventario,
    "consultar_demanda": consultar_demanda,
}


# ── 2 · Lo que el modelo ve de ellas. ────────────────────────────────────────
#
# ESTO es lo que viaja en la petición. El modelo no ve tu código: ve este JSON.
# Si tu descripción es mala, elige mal la herramienta — y no hay forma de que lo
# adivine. Escríbelo pensando en el modelo, no en un desarrollador.

ESQUEMAS = [
    {
        "type": "function",
        "function": {
            "name": "consultar_inventario",
            "description": "Cuántas toneladas de un material hay disponibles en una planta.",
            "parameters": {
                "type": "object",
                "properties": {
                    "planta": {"type": "string", "description": "Nombre de la planta, por ejemplo 'Itagüí'."},
                    "material": {"type": "string", "description": "Material: 'maíz' o 'soya'."},
                },
                "required": ["planta", "material"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_demanda",
            "description": "Cuántas toneladas de maíz necesita una planta esta semana.",
            "parameters": {
                "type": "object",
                "properties": {
                    "planta": {"type": "string", "description": "Nombre de la planta."},
                },
                "required": ["planta"],
            },
        },
    },
]

print(f"{len(FUNCIONES)} funciones · {len(ESQUEMAS)} esquemas declarados al modelo")

Ahora el bucle. Cuatro pasos, y **no hay ninguno más**:

1. Llamar al modelo declarando las herramientas.
2. ¿Pidió alguna? Ejecutarla.
3. Devolverle el resultado como un mensaje de rol `tool`.
4. Llamarlo otra vez para que responda.

Se detiene ahí: **una sola ronda**. Eso es lo que distingue a N3 de N4.

In [ ]:
NIVEL_ACTUAL = "N3"

SISTEMA_HERRAMIENTAS = (
    "Eres un asistente de operaciones. Usa las herramientas para obtener datos. "
    "NUNCA afirmes una cifra que no venga de una herramienta."
)


def n3(pregunta):
    mensajes = [
        {"role": "system", "content": SISTEMA_HERRAMIENTAS},
        {"role": "user", "content": pregunta},
    ]

    # PASO 1 · el modelo decide si quiere una herramienta
    paso("→ llamada 1 al modelo (con 2 herramientas declaradas)")
    respuesta = preguntar(mensajes, tools=ESQUEMAS, parallel_tool_calls=False)
    mensaje = respuesta.choices[0].message

    if not mensaje.tool_calls:
        paso("← respondió sin usar herramientas")
        return mensaje.content

    mensajes.append(mensaje)  # el historial tiene que incluir su petición

    for llamada in mensaje.tool_calls:
        # ─── ESTO es function calling. Mira el JSON, no el código. ───
        paso("← el modelo pide una herramienta. Su JSON, tal como llegó:")
        print(json.dumps(json.loads(llamada.function.arguments), indent=2, ensure_ascii=False))
        paso(f"  nombre: {llamada.function.name}")

        # PASO 2 · la ejecutas TÚ. El modelo no puede.
        argumentos = json.loads(llamada.function.arguments)
        resultado = FUNCIONES[llamada.function.name](**argumentos)
        paso(f"🔧 ejecutada → {resultado}")

        # PASO 3 · le devuelves el resultado
        mensajes.append({
            "role": "tool",
            "tool_call_id": llamada.id,
            "content": json.dumps(resultado, ensure_ascii=False),
        })

    # PASO 4 · y responde. Sin `tools=`: no queremos otra ronda.
    paso("→ llamada 2 al modelo (ya con el dato en el contexto)")
    final = preguntar(mensajes)
    return final.choices[0].message.content


print("RESPUESTA:", n3(PREGUNTA))

### 👀 Qué acaba de pasar

**Dos cosas, y las dos importan.**

**1. El JSON.** Eso es *function calling* completo. El modelo no tiene acceso a
`INVENTARIO`, no ejecutó nada: produjo un texto con la forma
`{"planta": "Itagüí", "material": "maíz"}` y tú lo convertiste en una llamada de
Python. Compáralo con la sección 7.2 del ebook: es la misma estructura.

**2. La respuesta está correcta y está incompleta.** Dijo cuánto maíz hay. No dijo
si alcanza — porque para eso necesitaba una segunda consulta y **no le dimos un
segundo turno**.

¿Qué le falta? Una palabra.

### 🔬 Tu turno

Quita `parallel_tool_calls=False` y vuelve a correr. El modelo pedirá las dos
herramientas de un golpe y responderá completo… y entonces N3 y N4 se vuelven
indistinguibles. Ese parámetro está ahí para que el límite del nivel se vea.

---

## 4 · N4 · Agente multipasos (ReAct) ★★★

```python
while llm_should_continue(): execute_next_step()
```

La respuesta a «¿qué le falta a N3?» es **iterar**. Y eso, en código, es un
`while`.

**Compara las dos funciones lado a lado.** La diferencia entre N3 y N4 es:

| | N3 | N4 |
|---|---|---|
| El bloque de llamada + ejecución | se hace **una vez** | va dentro de un `while` |
| Cómo termina | tras la segunda llamada | cuando el modelo **deja de pedir** herramientas |
| Protección | no hace falta | un tope de vueltas, o se cuelga |

Nada más. Ese es el salto de ★★☆ a ★★★.

In [ ]:
NIVEL_ACTUAL = "N4"


def n4(pregunta, esquemas=None, sistema=None, max_vueltas=6, sangria=0):
    """El mismo cuerpo de N3, dentro de un `while`.

    Los parámetros extra no son adorno: en N5 vamos a llamar a esta misma función
    con otro subconjunto de herramientas y otro prompt, y con eso tendremos dos
    especialistas sin escribir una línea nueva.
    """
    esquemas = esquemas or ESQUEMAS
    mensajes = [
        {"role": "system", "content": sistema or SISTEMA_HERRAMIENTAS},
        {"role": "user", "content": pregunta},
    ]

    for vuelta in range(1, max_vueltas + 1):          # ← EL BUCLE
        paso(f"→ vuelta {vuelta}", sangria)
        respuesta = preguntar(mensajes, tools=esquemas)
        mensaje = respuesta.choices[0].message

        if not mensaje.tool_calls:                     # ← LA SALIDA
            paso("← ya no pide herramientas: responde", sangria)
            return mensaje.content

        mensajes.append(mensaje)
        for llamada in mensaje.tool_calls:
            argumentos = json.loads(llamada.function.arguments)
            resultado = FUNCIONES[llamada.function.name](**argumentos)
            paso(f"🔧 {llamada.function.name}({argumentos}) → {resultado}", sangria)
            mensajes.append({
                "role": "tool",
                "tool_call_id": llamada.id,
                "content": json.dumps(resultado, ensure_ascii=False),
            })

    # Un agente sin tope no se equivoca: se cuelga. Y en clase eso es peor.
    paso("⚠ tope de vueltas alcanzado", sangria)
    return "No terminé dentro del límite de iteraciones."


print("RESPUESTA:", n4(PREGUNTA))

### 👀 Qué acaba de pasar

**Nadie le dijo que consultara dos cosas.** Decidió solo que necesitaba el
inventario y también la demanda, las comparó, y concluyó. Eso es agencia: no que
sea más listo, sino que **decide cuántos pasos dar**.

Y fíjate en el número de vueltas. Cada vuelta reenvía todo el historial al modelo,
así que **el contexto crece en cada iteración** — ahí está el costo, no en el
número de llamadas.

### 🔬 Tu turno

1. Pregúntale por una planta que no existe: `n4("¿Cuánto maíz hay en Cali?")`. La
   herramienta devuelve un `error` y el agente tiene que reportarlo. ¿Lo reporta o
   lo inventa? Esto cierra el arco que abrió N1.
2. Pon `max_vueltas=1` y vuelve a correr la pregunta original. Acabas de convertir
   N4 en N3.

### Y esto es lo que hace el framework

Las veinte líneas de arriba son, casi literalmente, lo que hay dentro de
`create_react_agent` de LangGraph:

```python
from langgraph.prebuilt import create_react_agent
agente = create_react_agent(modelo, herramientas, prompt=sistema)
agente.invoke({"messages": [("user", pregunta)]})
```

Tres líneas por veinte. Lo que el framework te quita de encima es el bucle, el
manejo del historial y los reintentos — **no el concepto.** Por eso lo escribimos
a mano primero: ahora, cuando veas esas tres líneas, sabes qué hay debajo.

---

## 5 · N5 · Sistema multiagente ★★★★

```python
if llm_trigger(): execute_agent()
```

Y acá viene la revelación del nivel: **no hay una arquitectura nueva.** Es N4
aplicado un nivel más arriba, en tres movimientos:

1. Llamar a `n4()` dos veces, cada vez con otro prompt y otro subconjunto de
   herramientas → dos especialistas.
2. Envolver cada uno en una función → **un agente expuesto como herramienta es
   solo una función que por dentro llama a un agente.**
3. Pasarle esas dos funciones como herramientas a un tercer `n4()`.

Para que haya dos dominios que separar, agregamos dos herramientas más.

In [ ]:
# ── Dos herramientas más, para que haya un segundo dominio ───────────────────

def historial_fallas(equipo):
    if equipo not in EQUIPOS:
        return {"error": f"No conozco el equipo {equipo}. Conozco: {list(EQUIPOS)}"}
    return {"equipo": equipo, **EQUIPOS[equipo]}


def estado_pedido(numero):
    if numero not in PEDIDOS:
        return {"error": f"No encuentro el pedido {numero}."}
    return {"numero": numero, **PEDIDOS[numero]}


FUNCIONES["historial_fallas"] = historial_fallas
FUNCIONES["estado_pedido"] = estado_pedido


def esquema(nombre, descripcion, **parametros):
    """Atajo para no repetir el JSON completo cuatro veces."""
    return {
        "type": "function",
        "function": {
            "name": nombre,
            "description": descripcion,
            "parameters": {
                "type": "object",
                "properties": {k: {"type": "string", "description": v}
                               for k, v in parametros.items()},
                "required": list(parametros),
            },
        },
    }


ESQUEMAS_OPERACIONES = [
    esquema("historial_fallas", "Fallas recientes y tendencia de vibración de un equipo.",
            equipo="Nombre del equipo, por ejemplo 'molino-1'."),
    esquema("estado_pedido", "En qué punto del proceso va un pedido y qué turno tiene.",
            numero="Número del pedido, por ejemplo 'PD-871'."),
]

print("abastecimiento:", [e["function"]["name"] for e in ESQUEMAS])
print("operaciones   :", [e["function"]["name"] for e in ESQUEMAS_OPERACIONES])

In [ ]:
NIVEL_ACTUAL = "N5"

# ── 1 y 2 · Dos especialistas. Cada uno es n4() con otro prompt y otras tools,
#            envuelto en una función para que el supervisor pueda llamarlo.

def agente_abastecimiento(instruccion):
    paso(f"⇒ delego en abastecimiento: «{instruccion}»")
    return n4(instruccion, ESQUEMAS,
              "Eres especialista en abastecimiento: inventarios y demanda. "
              "Responde denso y con cifras; te lee un supervisor, no un humano.",
              sangria=1)


def agente_operaciones(instruccion):
    paso(f"⇒ delego en operaciones: «{instruccion}»")
    return n4(instruccion, ESQUEMAS_OPERACIONES,
              "Eres especialista en mantenimiento y logística: equipos y pedidos. "
              "Responde denso y con cifras; te lee un supervisor, no un humano.",
              sangria=1)


# Los agentes entran al mismo registro de funciones que las herramientas normales,
# porque para el supervisor NO SON distintos: son funciones que se pueden llamar.
FUNCIONES["agente_abastecimiento"] = agente_abastecimiento
FUNCIONES["agente_operaciones"] = agente_operaciones

ESQUEMAS_AGENTES = [
    esquema("agente_abastecimiento",
            "Especialista en inventarios y demanda. Pásale una pregunta completa y "
            "autocontenida; devuelve un diagnóstico con cifras.",
            instruccion="La pregunta para el especialista, autocontenida."),
    esquema("agente_operaciones",
            "Especialista en equipos y pedidos. Pásale una pregunta completa y "
            "autocontenida; devuelve un diagnóstico con cifras.",
            instruccion="La pregunta para el especialista, autocontenida."),
]

SISTEMA_SUPERVISOR = (
    "Eres un supervisor de operaciones. NO consultas datos: coordinas a dos "
    "especialistas. Si la pregunta abarca los dos dominios, consulta a los dos. "
    "REGLA CRÍTICA: el especialista NO ve esta conversación. Copia en cada "
    "instrucción todos los datos que va a necesitar (nombre de la planta, número "
    "de pedido, nombre del equipo). Una instrucción incompleta lo deja sin poder "
    "consultar nada. "
    "Cierra con un diagnóstico único que diga qué dominio explica el problema y "
    "cuál no."
)


# ── 3 · El supervisor es... n4() otra vez, con los agentes como herramientas.
def n5(pregunta):
    return n4(pregunta, ESQUEMAS_AGENTES, SISTEMA_SUPERVISOR, max_vueltas=4)


# La pregunta nombra la planta a propósito: el supervisor tiene que pasarle ese
# dato a cada especialista, y si no lo tiene no puede.
PREGUNTA_CRUZADA = ("El pedido PD-871 de la planta de Itagüí va retrasado. ¿Es por "
                    "falta de materia prima o por un problema del molino-1?")

print("RESPUESTA:", n5(PREGUNTA_CRUZADA))

### 👀 Qué acaba de pasar

Mira la indentación de la salida: **hay un agente corriendo dentro de otro
agente.** El supervisor nunca tocó `INVENTARIO` ni `EQUIPOS` — repartió
instrucciones y sintetizó lo que le devolvieron.

Y mira `n5()`: es **una línea**. Todo el nivel salió de reutilizar `n4()` tres
veces. Eso es lo que la sección 8.1 del ebook llama *supervisor con llamada a
herramientas*, y no es una arquitectura nueva: es la misma de N4, un nivel arriba.

**Ahora la factura.** Corre la celda siguiente.

### 🔬 Tu turno

**1 · El costo de un supervisor que no hacía falta.** Hazle a N5 una pregunta de un
solo dominio y compárala con `n4()`:

```python
n5("¿Cuánto maíz hay en Buga?")
```

El multiagente se justifica cuando hay **separación real de dominios**, no por
sofisticación. Para una pregunta simple, un supervisor es un N4 caro.

**2 · El modo de falla del multiagente.** Quita la planta de la pregunta:

```python
n5("El pedido PD-871 va retrasado. ¿Es por materia prima o por el molino?")
```

Es muy probable que el supervisor le mande a abastecimiento algo como *«¿hay falta
de materia prima para el pedido PD-871?»* — sin decirle en qué planta. Y el
especialista **no ve la conversación con el usuario**: se queda sin poder consultar
nada y responde pidiendo el dato.

Eso es la falla característica de los sistemas multiagente y no tiene nada que ver
con la inteligencia del modelo: **el contexto no viaja gratis entre agentes.**
Cada frontera entre agentes es un sitio donde se pierde información, y hay que
escribirla a mano. Es la misma razón por la que N2 perdía la mitad del problema al
elegir un solo dominio.

---

## 6 · Cierre: la cuenta y la decisión

Los números de **tus** ejecuciones de hoy.

In [ ]:
from collections import defaultdict

resumen = defaultdict(lambda: {"llamadas": 0, "entrada": 0, "salida": 0})
for r in REGISTRO:
    d = resumen[r["nivel"]]
    d["llamadas"] += 1
    d["entrada"] += r["entrada"]
    d["salida"] += r["salida"]

base = resumen.get("N1", {}).get("entrada", 0) + resumen.get("N1", {}).get("salida", 0)

print(f"{'nivel':<7}{'llamadas':>9}{'tokens':>10}{'vs N1':>8}")
print("─" * 34)
for nivel in ["N1", "N2", "N3", "N4", "N5"]:
    if nivel not in resumen:
        continue
    d = resumen[nivel]
    tokens = d["entrada"] + d["salida"]
    rel = f"{tokens / base:.1f}×" if base else "—"
    print(f"{nivel:<7}{d['llamadas']:>9}{tokens:>10}{rel:>8}")

print()
if resumen.get("N2", {}).get("llamadas", 0) > 1:
    print(f"(N2 sale con {resumen['N2']['llamadas']} llamadas porque clasificamos "
          f"varias preguntas, no porque el nivel las necesite.)")
print("Cuenta las llamadas y cuenta los tokens. No crecen igual: lo que se dispara")
print("es el CONTEXTO que se reenvía en cada vuelta. Ahí está el costo real.")

### La tabla que hay que llevarse

| | N1 | N2 | N3 | N4 | N5 |
|---|---|---|---|---|---|
| Agencia | ☆☆☆ | ★☆☆ | ★★☆ | ★★★ | ★★★★ |
| Decide… | nada | qué ruta | qué herramienta | cuántos pasos dar | a qué agente |
| Consulta datos | no | no | 1 ronda | N rondas | vía especialistas |
| **Cuándo usarlo** | clasificar o redactar texto | triage, enrutamiento | consulta puntual | análisis multi-fuente | dominios separados |

**La última fila es el entregable de la sesión.** Cada Champion debería poder
señalar en qué columna cae su proyecto y defender por qué. Y ojo: *«el más alto»*
casi nunca es la respuesta correcta — una interfaz de consulta tipo aeropuerto es
un **N3 bien hecho**, y ponerle un supervisor sería pagar diez veces por lo mismo.

---

### Qué sigue

| | |
|---|---|
| **`2-los-cinco-niveles-taller.ipynb`** | Los mismos cinco niveles, pero los escribes tú: andamiaje con `# TODO` y `assert` que te dicen si quedó bien. Ese usa LangGraph y el mismo código que corre detrás del tablero. |
| **El tablero** · `localhost:8000` | El flujo dibujado: se ve la petición viajar por la arquitectura, entrar a la herramienta y volver. |
| **Tu entregable** | Escribe la herramienta que tu proyecto necesita, con su docstring pensada para el modelo, y conéctala a `n4()`. Si el agente la ignora, la sospechosa es la descripción. |

<div align="center">
  <strong>Qypher · Formación en Inteligencia Artificial</strong>
</div>